# Pandas on the Titanic dataset

This tutorial introduces the different mechanisms of `pandas` using the [Titanic dataset](https://www.openml.org/d/40945) (a classic).

In [ ]:
!wget https://www.openml.org/data/get_csv/16826755/phpMYEkMl
!mv phpMYEkMl titanic.csv
import pandas as pd
import matplotlib.pyplot as plt

## Loading data

* Start by loading the data from the `titanic.csv` file using the command [`pandas.read_csv`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html) into a new `DataFrame`.
* Display some rows of the `DataFrame`.
* Display the general information of the `DataFrame` with [`pandas.DataFrame.describe`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.describe.html)

A list of the columns with a short description :
* survived - Survival (0 = No; 1 = Yes)
* pclass - Passenger Class (1 = 1st; 2 = 2nd; 3 = 3rd)
* name - Name
* sex - Sex
* age - Age
* sibsp - Number of Siblings/Spouses Aboard
* parch - Number of Parents/Children Aboard
* ticket - Ticket Number
* fare - Passenger Fare
* cabin - Cabin
* embarked - Port of Embarkation (C = Cherbourg; Q = Queenstown; S = Southampton)
* boat - Lifeboat (if survived)
* body - Body number (if did not survive and body was recovered)

In [ ]:
# df = ???

### Solution

In [ ]:
# Loading data
df = pd.read_csv("titanic.csv")

# Dataframe display
df

In [ ]:
df.describe()

## Detect columns containing `"?"`

Some values are incomplete and contain the value `"?"` They need to be replaced.

To begin with, identify all the columns that contain a `"?"`. Use the appproach you are the most confortable with. You can for example use the function [`pandas.Series.unique`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.unique.html) on each column and isolate the one containing a `"?"`

In [ ]:
# Your code here

### Solution

In [ ]:
# Using filtering
count_missing = (df == "?").sum()
cols_to_fill = set(count_missing[count_missing > 0].index)
print("Columns with at least one ?")
print(cols_to_fill)

print("--------------")
# Using unique()
cols_to_fill = set()
for c in df.columns:
  if df[c].dtype == "O" and "?" in df[c].unique():
    cols_to_fill.add(c)
print("Columns with at least one ?")
print(cols_to_fill)

## Data replacement

Replace the data in columns containing `"?"` with the following values:
* **`0`** for `age`, `fare` and `body`
* The string **`unknown`** for the other columns: `embarked`, `home.dest`, `cabin` and `boat`

To change the values you can use `df.loc[filter, col] = new_val`.

In [ ]:
# # Replacement of numerical values
# ???

# # Replacement of nominal values
# ???

# # Specifying the new column types
# new_types={"fare": "float32",
#            "age": "float32",
#            "body": "int64"}
# df = df.astype(new_types)

## Solution

In [ ]:
# Replacement of numerical values
for col in {"age", "fare", "body"}:
  df.loc[df[col] == "?", col] = 0

# Replacement of nominal values
for  col in {'boat', 'cabin', 'embarked', 'home.dest'}:
  df.loc[df[col] == "?", col]="unknown"

# Specifying the new column types
new_types={"fare": "float32",
           "age": "float32",
           "body": "int64"}
df = df.astype(new_types)

## A first calculation

Pandas should now have no secrets for you.

Calculate the percentage of women and men who survived the sinking in proportion to their sex:

$$
  \frac{\text{Man}_{\text{survived}}}{\text{Man}_{\text{total}}} \, \text{and} \, \frac{\text{Woman}_{\text{survived}}}{\text{Woman}_{\text{total}}}
$$

In [ ]:
# Your code here

### Solution

In [ ]:
n_surv = df[df.survived == 1].sex.value_counts()
n_pass = df.sex.value_counts()
p_surv = n_surv / n_pass

print(f"Among the {n_pass['male']} men on board, "
      f"{n_surv['male']} survived. Which is {p_surv['male']:.2%} of all men on board")
print(f"Among the {n_pass['female']} women on board, "
      f"{n_surv['female']} survived. Which is {p_surv['female']:.2%} of all women on board")

## A simple visualization

Create a barplot displaying the average ticket price according to the class of the passenger and whether he survived.

You are free to split the layout into two `Axes` or one.

In [ ]:
# Your code here

### Solution

In [ ]:
width = 0.35

fig, ax = plt.subplots()
ax.bar(df[df.survived == 1].pclass - width / 2,
       df[df.survived == 1].fare,
       width)
ax.bar(df[df.survived == 0].pclass + width / 2,
       df[df.survived == 0].fare,
       width)

ax.legend(["Survived", "Deceased"])
ax.set_xticks([1, 2, 3])
ax.set_xticklabels(["$1^{re}$", "$2^{de}$", "$3^e$"])
ax.set_xlabel("Class")
ax.set_ylabel("Ticket fare")

plt.show()

## New column `first_name`

Add a `first_name` column to your DataFrame that contains the passenger's first name:
* Use the `name` column
* Extract the first name from it using [`pandas.Series.str.split`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.str.split.html). The first name is always displayed before the comma `,` in `name`.
* Add the result to the `first_name` column


In [ ]:
# Your code here

### Solution

In [ ]:
df["first_name"] = [fullname[0] for fullname in df.name.str.split(",")]
df

## Mapping

Using [`pandas.Series.apply`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.map.html), change the first names in the `first_name` column into uppercase by mapping the [`str.upper`](https://docs.python.org/fr/3/library/stdtypes.html#str.upper) function from the [standard library](https://docs.python.org/fr/3/library/stdtypes.html) onto the entire column.

In [ ]:
# Your code here

### Solution

In [ ]:
df["first_name"] = df["first_name"].apply(str.upper)
df["first_name"]

## Average age per class


In this exercise, you will need to calculate the average age per class and add it to a new `age_per_class` column in the DataFrame. This is obviously not ideal since we are introducing redundancy into the data, but the goal is to use the methods we have already seen and then practice the methods [`pandas.DataFrame.groupby`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.groupby.html) and [`pandas.DataFrame.join`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.join.html).

1. First, use all the previous techniques at your disposal to create the new column (filters, ...):
 * Initialize the new column with a default value
 * Do a loop on the classes and calculate the average per class, then add it by filtering
2. In a second step, you should try the same exercise again but without looping and using only [`pandas.DataFrame.groupby`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.groupby.html) and [`pandas.DataFrame.join`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.join.html):
 * Make a `DataFrame` by computing a groupby on `pclass` with averaging
 * Compute the join between your initial `DataFrame` and your groupby.

In [ ]:
# # Computation of the average age per class using a loop
# df["age_per_class"] = # default value
# for v in df.pclass.unique():
#   ???

# print("RESULT USING A LOOP")
# print(df.age_per_class)
# print()

In [ ]:
# # Use this line to suppress the newly created column and restart with group_by
# df = df.drop("age_per_class", axis=1)

In [ ]:
# # Using group_by
#
# # group_by computation
# gb_pclass=
# # join
# df = df.join(???)

# print("RESULT USING GROUP_BY")
# print(df.age_per_class)
# print()

### Solution

In [ ]:
# Computation of the average age per class using a loop
mean_age = df.age.mean()

df["age_per_class"] = mean_age
for v in df.pclass.unique():
  df.loc[df.pclass == v, "age_per_class"] = df[df.pclass == v].age.mean()

print("RESULT USING A LOOP")
print(df.age_per_class)
print()

In [ ]:
# Use this line to suppress the newly created column and restart with group_by
df = df.drop("age_per_class", axis=1)

In [ ]:
# Utilisation group_by
gb_pclass = df.groupby("pclass").mean()
df = df.join(gb_pclass["age"], on="pclass", how="left", rsuffix="_per_class")

print("RESULT USING GROUP_BY")
print(df.age_per_class)
print()